[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_3_camera_model/06_camera_model_theory.ipynb)

# Notebook 06 — Camera Model Theory
### 6D Pose Vision Workshop · Part 3: Camera Model

**Estimated time:** 60 minutes  
**Dependencies:** `numpy`, `matplotlib`

---

## Recommended Watch

> Watch this **before** opening the notebook — it gives you the visual intuition for the pinhole model, intrinsic matrix, and why calibration matters.

| Title | Link | Duration |
|---|---|---|
| **Camera Calibration Explained and SIMPLE Step-by-Step Guide!** | [▶ Watch](https://www.youtube.com/watch?v=Wcnb197g2i0) | ~15 min |

---

> **Optional — want the deep theory?**
> Dr. Shree K. Nayar (Columbia University) covers image formation from scratch with very visual examples. These are lecture-style videos — watch at your own pace, as many or as few as you want.

| Title | Link |
|---|---|
| **Image Formation \| Image Sensing \| Binary Images** *(playlist)* | [▶ Playlist](https://youtube.com/playlist?list=PL2zRqk16wsdr9X5rgF-d0pkzPdkHZ4KiT&si=oikb1sr5LxefxFJG) |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib ipywidgets -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
print(f"NumPy: {np.__version__}")

## Learning Objectives

By the end of this notebook you will:

- Understand the **three coordinate frames**: world, camera, image
- Know the **pinhole camera model** and why it's a useful approximation
- Read and build the **intrinsic matrix K** — focal lengths and principal point
- Understand **extrinsics** `[R|t]` — where the camera is in the world
- Apply the full **projection equation** $\mathbf{x} = K[R|t]\mathbf{X}$
- Understand **homogeneous coordinates** and why we need them

---

## 1. The Three Coordinate Frames

Every camera system involves three distinct coordinate systems. Confusing them is the #1 source of bugs in pose estimation.

### World Frame (3D)

A fixed reference frame in the physical world. You define it — usually at the center of the chessboard, the corner of a room, or wherever is convenient.

- Points: $\mathbf{X}_w = (X_w, Y_w, Z_w)$
- Units: meters (or mm, cm — just be consistent)

### Camera Frame (3D)

Attached to the camera itself. The camera's optical center is the origin. Z points forward (into the scene), X points right, Y points down.

- Points: $\mathbf{X}_c = (X_c, Y_c, Z_c)$
- Related to world frame by extrinsics [R|t]

### Image Frame (2D)

The 2D pixel grid on the camera sensor.

- Points: $\mathbf{x} = (u, v)$ in pixels
- Origin: top-left corner (in OpenCV convention)

```
                    Z_c (camera looks along +Z)
                    |
   [WORLD]          |   [CAMERA FRAME]         [IMAGE]
                    |                           ┌─────────► u (cols, right)
   Z_w   Y_w        +───── X_c (right)          │
    |   /           |                           │
    |  /            Y_c (down)                  │
    | /                                         ▼ v (rows, down)
    +─────── X_w

   World Frame       Camera Frame              Image Frame
   (you define)      (fixed to camera)         (pixels)
```

<img src="https://lh3.googleusercontent.com/d/1A7FCuofdpeO3zBGS_ik01ml_fFzcVnIb" width="700" alt="Three coordinate frames diagram"/>

### The pipeline

$$\underbrace{\mathbf{X}_w}_{\text{world}} \overset{[R|t]}{\longrightarrow} \underbrace{\mathbf{X}_c}_{\text{camera}} \overset{K}{\longrightarrow} \underbrace{\mathbf{x}}_{\text{image}}$$

**Extrinsics** $[R|t]$: world → camera  
**Intrinsics** $K$: camera → image

<img src="https://lh3.googleusercontent.com/d/1NGYT7UEBUg-BfgdHkVAgpnPp2wlVMlRS" width="700" alt="Camera projection pipeline diagram"/>

---

## 2. The Pinhole Camera Model

### The physical intuition

Imagine a box with a tiny pinhole on one side and a screen on the other. Light from a 3D scene passes through the pinhole and forms an image on the screen. This is the **pinhole camera model**.

<img src="https://lh3.googleusercontent.com/d/1l8vnEHuhE2OSKW1adkLLOnEQ_12BmomA" width="700" alt="Pinhole camera model diagram"/>

The **focal length** $f$ is the distance from the pinhole to the image plane.

### The projection math

How does a 3D point become a 2D pixel? The answer is **similar triangles** — pure geometry, no complicated optics.

Look at the scene from the side (the $X_c$–$Z_c$ plane). You have a 3D point $P$ at position $(X_c, Z_c)$, a pinhole at the origin, and an image plane at distance $f$. A ray of light travels in a straight line from $P$, through the pinhole, and hits the image plane at $x'$. That one straight line creates **two similar triangles** sharing the same angle at the pinhole:

<img src="https://lh3.googleusercontent.com/d/1HpBuC6ZkQDv-xpctE-KFEqURCQLE2UpH" width="600" alt="Similar triangles derivation diagram"/>

**Big triangle** (pinhole → P): base $Z_c$, height $X_c$

**Small triangle** (pinhole → image): base $f$, height $x'$

Both triangles share the same angle at the pinhole, so their sides are proportional:

$$\frac{x'}{f} = \frac{X_c}{Z_c} \qquad \Rightarrow \qquad x' = f \cdot \frac{X_c}{Z_c}$$

The same argument applied to the $Y_c$–$Z_c$ plane gives the vertical coordinate. The resulting projection formulas are:

$$\boxed{x' = f \cdot \frac{X_c}{Z_c} \qquad y' = f \cdot \frac{Y_c}{Z_c}}$$

**Annotation:**
- $X_c, Y_c$ — lateral position of the point (meters)
- $Z_c$ — depth of the point (how far it is from the camera, meters)
- $f$ — focal length (meters or pixels)
- $x', y'$ — projected position on image plane

**Key insight:** Dividing by $Z_c$ is perspective division — objects farther away appear smaller because $Z_c$ is larger.

<img src="https://lh3.googleusercontent.com/d/1B1Pi99YvEhxGnk7GqAuVb6uby3slmIO2" width="700" alt="Perspective division and viewing frustum"/>

**The viewing frustum.** The pyramid-shaped volume you see above is called the **viewing frustum** — it is the region of space a camera can actually see. It is the direct physical consequence of applying perspective division to the camera's field of view.

Notice that dividing by $Z_c$ breaks down as $Z_c \to 0$: the result shoots to infinity. This is exactly why a frustum has a **Near Plane** — by cutting off the tip of the pyramid we guarantee $Z_c$ is never zero or dangerously small, so the math stays stable. The Far Plane cuts the other end to keep depth buffers precise. In short: perspective division is the formula, and the viewing frustum is the region of space where that formula can safely run.

### From metric to pixels

The image plane is measured in physical units (mm), but images are measured in pixels. Scaling factors $f_x$ and $f_y$ (focal lengths in **pixels**) absorb this conversion:

$$u = f_x \cdot \frac{X_c}{Z_c} + c_x \qquad v = f_y \cdot \frac{Y_c}{Z_c} + c_y$$

- $f_x, f_y$ — focal length in pixels (typically same if pixels are square)
- $c_x, c_y$ — principal point: the pixel where the optical axis pierces the image (usually near the image center)

<img src="https://lh3.googleusercontent.com/d/1hlT75quoyLi-4kWf19LYINC_rw-zpFhi" width="700" alt="From metric to pixels diagram"/>

### Want to go further? Other projection models

This notebook focuses on **perspective projection** (the pinhole model), but it is one of several projection types you will encounter in computer vision, 3D graphics, and robotics:

| Projection | Key idea | Where you see it |
|---|---|---|
| **Perspective** *(this notebook)* | Divides by $Z_c$ — near objects appear larger | Real cameras, OpenCV, most CV algorithms |
| **Orthographic** | No depth division — parallel lines stay parallel | CAD software, engineering drawings, some AR overlays |
| **Fisheye / Equidistant** | Maps a very wide FOV onto the sensor with a non-linear model | Drone cameras, 360° cameras, SLAM |
| **Stereographic** | Preserves angles at the cost of area — used in wide-FOV lenses | Architectural photography, some robotics cameras |
| **Cylindrical / Spherical** | Projects onto a cylinder or sphere — useful for panoramas | Photogrammetry, VR headsets, Google Street View |

If you ever need to search these topics: look for **camera projection models**, **lens distortion models**, or **non-perspective cameras**.

---

In [ ]:
# Visualize the perspective projection effect

# Camera parameters (typical webcam)
fx, fy = 800, 800   # focal length in pixels
cx, cy = 320, 240   # principal point (image center for 640x480)

def project_point(X_c, Y_c, Z_c, fx, fy, cx, cy):
    """Project a 3D camera-frame point to 2D image pixel."""
    if Z_c <= 0:
        return None  # behind camera
    u = fx * (X_c / Z_c) + cx
    v = fy * (Y_c / Z_c) + cy
    return u, v

# A real object is 0.1m tall (10cm), at different depths
object_top    = 0.0   # top of object in Y_c
object_bottom = 0.1   # 10cm below
object_X      = 0.0   # centered

depths = [0.5, 1.0, 2.0, 4.0]  # meters from camera

print("Perspective effect: same 10cm object at different depths")
print(f"{'Depth (m)':<12} {'Projected top v':<18} {'Projected bot v':<18} {'Height (px)':<12}")
print("-" * 60)

for Z in depths:
    top_pix = project_point(object_X, object_top, Z, fx, fy, cx, cy)
    bot_pix = project_point(object_X, object_bottom, Z, fx, fy, cx, cy)
    height_px = abs(bot_pix[1] - top_pix[1])
    print(f"{Z:<12.1f} {top_pix[1]:<18.1f} {bot_pix[1]:<18.1f} {height_px:<12.1f}")

print("\n→ Object appears half as tall when distance doubles (inverse depth relationship)")

## 3. The Intrinsic Matrix K

### Building K from the projection equations

We can write the pixel projection equations in matrix form using **homogeneous coordinates** (adding a 1 to work with matrix multiplication):

$$\begin{pmatrix} u \\ v \\ 1 \end{pmatrix} = \frac{1}{Z_c} \underbrace{\begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}}_{\mathbf{K}} \begin{pmatrix} X_c \\ Y_c \\ Z_c \end{pmatrix}$$

**Annotation of K:**

$$\mathbf{K} = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

| Parameter | Meaning | Typical value (640×480 webcam) |
|---|---|---|
| $f_x$ | Focal length in pixels (horizontal) | 600–1200 px |
| $f_y$ | Focal length in pixels (vertical) | ≈ $f_x$ for square pixels |
| $c_x$ | Principal point x — usually near width/2 | ~320 px |
| $c_y$ | Principal point y — usually near height/2 | ~240 px |

### Why is the off-diagonal zero?

The zero at K[0,1] is the **skew coefficient** — it would be nonzero only if the pixel axes are not perpendicular. In modern cameras, this is always effectively zero.

### What does K depend on?

K depends **only on the camera itself** (lens, sensor size, resolution) — not on where the camera is pointed. It does NOT change when you move or rotate the camera. Once calibrated, K is fixed until you change the lens or zoom.

### Relationship between focal length and field of view

$$\text{FOV}_x = 2 \arctan\left(\frac{W}{2 f_x}\right)$$

Where $W$ is the image width in pixels. Larger $f_x$ → smaller field of view (telephoto). Smaller $f_x$ → wider field of view (wide angle).

<img src="https://lh3.googleusercontent.com/d/1xsE24YjNyl3HEFlVXfmzHI_2dfyKc0PL" width="700" alt="Focal length and field of view relationship"/>

In [ ]:
# Build and inspect the intrinsic matrix K

def build_K(fx, fy, cx, cy):
    """Build the 3x3 camera intrinsic matrix."""
    return np.array([[fx,  0,  cx],
                     [ 0, fy,  cy],
                     [ 0,  0,   1]], dtype=np.float64)

def fov_from_K(K, width, height):
    """Compute horizontal and vertical FOV from K."""
    fov_x = 2 * np.degrees(np.arctan(width  / (2 * K[0, 0])))
    fov_y = 2 * np.degrees(np.arctan(height / (2 * K[1, 1])))
    return fov_x, fov_y

# Three example cameras
cameras = {
    'Webcam 640x480':     build_K(fx=600,  fy=600,  cx=320, cy=240),
    'Phone 1080p':        build_K(fx=1400, fy=1400, cx=540, cy=960),
    'Wide-angle 640x480': build_K(fx=300,  fy=300,  cx=320, cy=240),
}
resolutions = [(640, 480), (1080, 1920), (640, 480)]

for (name, K), (w, h) in zip(cameras.items(), resolutions):
    fov_x, fov_y = fov_from_K(K, w, h)
    print(f"\n{name}:")
    print(f"  K =\n{K}")
    print(f"  FOV: {fov_x:.1f}° (horizontal) × {fov_y:.1f}° (vertical)")

## 4. Homogeneous Coordinates — Why We Need Them

### The problem with regular coordinates

Projection (dividing by $Z_c$) is a **nonlinear** operation — you can't represent it as a simple matrix multiplication in regular coordinates. This makes chaining transformations awkward.

### The solution: add a 1

In **homogeneous coordinates** we represent a point $(x, y)$ as $(x, y, 1)$ — or more generally $(wx, wy, w)$ for any nonzero $w$. Converting back: divide by the last element.

$$\begin{pmatrix} x \\ y \\ 1 \end{pmatrix} \overset{\text{homogeneous}}{\longleftrightarrow} (x, y) \qquad \begin{pmatrix} wx \\ wy \\ w \end{pmatrix} \overset{\text{homogeneous}}{\longleftrightarrow} (x, y)$$

For 3D: $(X, Y, Z) \leftrightarrow (X, Y, Z, 1)$

### Why this helps

With homogeneous coordinates, perspective projection becomes a **matrix multiplication** (linear!):

$$\tilde{\mathbf{x}} = K [R|t] \tilde{\mathbf{X}}$$

Where $\tilde{\mathbf{x}}$ is the homogeneous image point and $\tilde{\mathbf{X}}$ is the homogeneous world point. After the multiplication, divide by the last element to get pixel coordinates.

In [ ]:
# Demonstrate homogeneous coordinates for projection

def project_homogeneous(X_world, K, R, t):
    """
    Project 3D world point to 2D image pixel using x = K[R|t]X.
    
    Args:
        X_world: (3,) array — world point
        K: (3,3) intrinsic matrix
        R: (3,3) rotation matrix
        t: (3,) translation vector
    Returns:
        (u, v) pixel coordinates
    """
    # Build [R|t] — the 3x4 extrinsic matrix
    Rt = np.hstack([R, t.reshape(3, 1)])  # shape (3, 4)
    
    # Convert world point to homogeneous: (X, Y, Z) → (X, Y, Z, 1)
    X_h = np.append(X_world, 1.0)  # shape (4,)
    
    # Full projection: P = K @ [R|t]  (shape 3x4)
    P = K @ Rt  # shape (3, 4)
    
    # Project: x_h = P @ X_h  → shape (3,)
    x_h = P @ X_h
    
    # Convert from homogeneous: divide by last element
    u = x_h[0] / x_h[2]
    v = x_h[1] / x_h[2]
    
    return u, v, x_h, P

# Camera setup
K = build_K(fx=800, fy=800, cx=320, cy=240)

# Identity rotation (camera aligned with world), camera at origin
R = np.eye(3)
t = np.array([0.0, 0.0, 0.0])

# Test points in front of the camera (Z > 0 = in front)
test_points = [
    np.array([0.0,  0.0,  1.0]),   # straight ahead, 1m
    np.array([0.5,  0.0,  1.0]),   # 0.5m to the right at 1m depth
    np.array([0.0,  0.5,  2.0]),   # 0.5m down at 2m depth
    np.array([-0.3, -0.2, 0.8]),   # left-up at 0.8m
]

print(f"K =\n{K}\n")
print(f"{'World point':<30} {'Pixel (u, v)':<20} {'Depth Z_c':<12}")
print("-" * 62)

for X in test_points:
    u, v, x_h, P = project_homogeneous(X, K, R, t)
    print(f"({X[0]:+.2f}, {X[1]:+.2f}, {X[2]:+.2f})       "
          f"({u:6.1f}, {v:6.1f})     {X[2]:.2f} m")

## 5. Extrinsics — [R|t]

### What extrinsics encode

<img src="https://lh3.googleusercontent.com/d/1MDYJ1kbetpCZbjmcheE92_nvs4kooZn4" width="700" alt="Extrinsics: camera position and orientation relative to world frame"/>

The extrinsic parameters describe **where the camera is and how it's oriented** relative to the world frame:

- $\mathbf{R}$ (rotation matrix, $3 \times 3$): describes the camera's orientation
- $\mathbf{t}$ (translation vector, $3 \times 1$): describes the camera's position

Together, they transform world coordinates to camera coordinates:

$$\mathbf{X}_c = R \mathbf{X}_w + \mathbf{t}$$

<img src="https://lh3.googleusercontent.com/d/14ZxZkXjmyVtRFEgrEFN8Wnt1VLuZS9GP" width="700" alt="World to camera coordinate transformation"/>

### Rotation matrix properties

A valid rotation matrix must satisfy:

$$R^T R = I \quad \text{(orthogonal)} \qquad \det(R) = +1 \quad \text{(proper rotation, not reflection)}$$

These 9 numbers encode 3 degrees of freedom (3D orientation). The constraints reduce 9 → 3.

### The 4×4 homogeneous transform matrix

It's convenient to package $[R|t]$ into a $4 \times 4$ matrix:

$$\mathbf{T} = \begin{bmatrix} R & t \\ 0^T & 1 \end{bmatrix} = \begin{bmatrix} r_{11} & r_{12} & r_{13} & t_x \\ r_{21} & r_{22} & r_{23} & t_y \\ r_{31} & r_{32} & r_{33} & t_z \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

This form lets rotation and translation be applied in a single matrix multiply — useful when working with homogeneous coordinates (Section 4).

### Rodrigues rotation vector (rvec)

OpenCV represents rotation as a **3-element vector** `rvec` instead of a 3×3 matrix:

- Direction of `rvec` = axis of rotation
- Magnitude `||rvec||` = angle of rotation in radians

$$R = \text{Rodrigues}(\mathbf{rvec}) \quad \leftrightarrow \quad \mathbf{rvec} = \text{Rodrigues}(R)$$

This is the **Rodrigues formula** — OpenCV's `cv2.Rodrigues()` converts between the two.

<img src="https://lh3.googleusercontent.com/d/17vKSdcRuZJXbKcrKWPKJKuo02ScDQywf" width="700" alt="Rodrigues rotation vector visualization"/>

In [ ]:
import cv2

# Demonstrate rotation matrices and Rodrigues conversion

def rotation_x(theta_deg):
    """Rotation matrix around X axis by theta degrees."""
    theta = np.radians(theta_deg)
    return np.array([[1, 0, 0],
                     [0, np.cos(theta), -np.sin(theta)],
                     [0, np.sin(theta),  np.cos(theta)]])

def rotation_y(theta_deg):
    """Rotation matrix around Y axis by theta degrees."""
    theta = np.radians(theta_deg)
    return np.array([[ np.cos(theta), 0, np.sin(theta)],
                     [ 0,             1, 0            ],
                     [-np.sin(theta), 0, np.cos(theta)]])

def rotation_z(theta_deg):
    """Rotation matrix around Z axis by theta degrees."""
    theta = np.radians(theta_deg)
    return np.array([[np.cos(theta), -np.sin(theta), 0],
                     [np.sin(theta),  np.cos(theta), 0],
                     [0,              0,             1]])

# Example: camera tilted 30° downward (rotating around X axis)
R = rotation_x(30)
print("Rotation matrix (30° tilt around X — camera looks slightly down):")
print(np.round(R, 4))

# Verify orthogonality
print(f"\nR.T @ R = I? {np.allclose(R.T @ R, np.eye(3), atol=1e-10)}")
print(f"det(R) = {np.linalg.det(R):.6f}  (should be +1.0)")

# Convert to Rodrigues vector and back
rvec, _ = cv2.Rodrigues(R)
R_back, _ = cv2.Rodrigues(rvec)
print(f"\nRodrigues rvec:\n {rvec}")
print(f"||rvec|| = {np.linalg.norm(rvec):.4f} rad = {np.degrees(np.linalg.norm(rvec)):.1f}°")
print(f"Round-trip: R → rvec → R matches? {np.allclose(R, R_back, atol=1e-10)}")

## 6. The Full Projection Equation

### Putting it all together

$$\underbrace{\lambda \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}}_{\text{image point}} = \underbrace{\begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}}_{K} \underbrace{\begin{bmatrix} r_{11} & r_{12} & r_{13} & t_x \\ r_{21} & r_{22} & r_{23} & t_y \\ r_{31} & r_{32} & r_{33} & t_z \end{bmatrix}}_{[R|t]} \underbrace{\begin{pmatrix} X_w \\ Y_w \\ Z_w \\ 1 \end{pmatrix}}_{\text{world point}}$$

Where $\lambda = Z_c$ (the depth in camera frame — the perspective division factor).

**Compact form:** $\mathbf{x} = K [R|t] \mathbf{X}$

<img src="https://lh3.googleusercontent.com/d/1Mn_Nrzyr6gZN97GaZRVXXeO88_HjMcg0" width="700" alt="Full projection pipeline diagram"/>

### What the demo below does

The code cell below makes this equation concrete. Here's what it does:

1. **Build a virtual camera** — define `K` (focal length, principal point) and `[R|t]` (where the camera sits and how it's tilted). This is your camera on paper.
2. **Build a virtual 3D cube** — 8 corners placed in world coordinates. Nothing rendered, just numbers: $(X_w, Y_w, Z_w)$ for each corner.
3. **Apply the projection equation manually** — for each corner, compute `X_cam = R @ corner + t` (extrinsics, world → camera), then `u = fx * X_cam[0]/X_cam[2] + cx` (intrinsics, camera → pixel). This is $\mathbf{x} = K[R|t]\mathbf{X}$ done by hand.
4. **Plot both sides** — left plot: the 3D cube in world space with the camera position. Right plot: the resulting 2D pixel coordinates — what that camera would actually see on its sensor.

**Goal:** you just learned $\mathbf{x} = K[R|t]\mathbf{X}$ in theory. This demo runs that equation on a real shape so you can see the math produce a recognizable perspective projection — closer edges appear bigger, farther edges smaller.

In [ ]:
try:
    import ipywidgets as widgets
    from ipywidgets import interact
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found — run: pip install ipywidgets")

# Fixed scene setup
R_demo = rotation_x(15)
t_demo = np.array([0.0, -0.15, 0.5])   # camera 0.5m away, slightly up

cube_corners_world = np.array([
    [0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0],  # bottom face
    [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1],  # top face
], dtype=np.float64) * 0.3  # 30cm cube

edges_bottom = [(0,1),(1,2),(2,3),(3,0)]
edges_top    = [(4,5),(5,6),(6,7),(7,4)]
edges_vert   = [(0,4),(1,5),(2,6),(3,7)]
all_edges    = edges_bottom + edges_top + edges_vert

W, H = 640, 480
cx, cy = W // 2, H // 2

def project_and_draw(fov_deg=60):
    plt.close('all')

    # Focal length derived from FOV: fx = (W/2) / tan(FOV/2)
    fx = fy = (W / 2) / np.tan(np.radians(fov_deg / 2))
    K = build_K(fx, fy, cx, cy)

    # Project cube corners
    projected = []
    for corner in cube_corners_world:
        X_cam = R_demo @ corner + t_demo
        u = fx * X_cam[0] / X_cam[2] + cx
        v = fy * X_cam[1] / X_cam[2] + cy
        projected.append((u, v))
    projected = np.array(projected)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- 3D world view (static — shows the cube and camera position) ---
    ax1 = fig.add_subplot(121, projection='3d')
    for i, j in all_edges:
        pts = cube_corners_world[[i, j]]
        ax1.plot(pts[:,0], pts[:,1], pts[:,2], 'b-', alpha=0.7)
    ax1.scatter(*cube_corners_world.T, c='blue', s=40)
    cam_pos = -R_demo.T @ t_demo
    ax1.scatter(*cam_pos, c='red', s=100, marker='^', label='Camera')
    all_pts = np.vstack([cube_corners_world, cam_pos])
    mid = all_pts.mean(axis=0)
    half_range = np.abs(all_pts - mid).max() * 1.1
    ax1.set_xlim(mid[0]-half_range, mid[0]+half_range)
    ax1.set_ylim(mid[1]-half_range, mid[1]+half_range)
    ax1.set_zlim(mid[2]-half_range, mid[2]+half_range)
    ax1.set_box_aspect([1, 1, 1])
    ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)'); ax1.set_zlabel('Z (m)')
    ax1.set_title('3D World Frame')
    ax1.legend()

    # --- 2D projected image (updates with FOV slider) ---
    ax2 = fig.add_subplot(122)
    ax2.set_facecolor('#222')
    for i, j in all_edges:
        pts = projected[[i, j]]
        ax2.plot(pts[:,0], pts[:,1], 'cyan', linewidth=2, alpha=0.8)
    ax2.scatter(projected[:,0], projected[:,1], c='cyan', s=50, zorder=5)
    ax2.axhline(cy, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    ax2.axvline(cx, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    ax2.set_xlim(0, W); ax2.set_ylim(H, 0)
    ax2.set_aspect('equal')
    ax2.set_xlabel('u (pixels)'); ax2.set_ylabel('v (pixels)')
    ax2.set_title(f'Camera sees the cube  |  FOV={fov_deg}°  |  fx={fx:.0f}px')

    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    interact(project_and_draw,
             fov_deg=widgets.IntSlider(min=20, max=120, step=5, value=60,
                                       description='FOV (°)',
                                       style={'description_width': 'initial'}))
else:
    project_and_draw(fov_deg=60)

### Is the projection matrix invertible?

Good question — and the answer is **no, not directly**. Here's why.

The projection $\mathbf{x} = K[R|t]\mathbf{X}$ takes a 3D point and collapses it to 2D. During that collapse, **depth is lost**: a point 1m away and a point 10m away along the same line of sight both map to the exact same pixel. Many 3D points → one pixel. That's a many-to-one operation, and many-to-one operations can't be inverted.

If you try to "go back" from a single pixel $(u, v)$ to 3D space, the best you can do is recover a **ray** — an infinite line through the camera origin in the direction of that pixel. Every point along that ray projects to the same pixel. You cannot know *where* on the ray the actual 3D point lives without more information.

```
             camera
               *
              /|\ 
             / | \      All three points project
            /  |  \     to the same pixel (u, v)
           P1  P2  P3
```

### So how do you recover 3D from 2D?

You need an extra constraint. There are three common strategies:

| Strategy | How it works | Example |
|---|---|---|
| **Known depth** | Add a depth sensor — directly measure $Z_c$ | RGB-D cameras (Intel RealSense) |
| **Multiple views** | Two cameras see the same point → triangulate | Stereo vision, Structure from Motion |
| **Known 3D geometry** | You already have the 3D model — match model points to image points | Pose estimation with solvePnP |

### What 6D pose estimation does — the real story

In 6D pose estimation, you already have the 3D object model. You know exactly where each point on that object lives in 3D space (world coordinates). What you *don't* know is how the camera is positioned relative to the object — i.e., $R$ and $t$.

So you flip the problem: instead of asking *"where does this 3D point land in the image?"*, you ask:

> **"Given that I can see these 2D pixels in the image, and I know the corresponding 3D points on the object — what must $R$ and $t$ be?"**

Each 2D–3D correspondence gives you 2 equations (one for $u$, one for $v$). $R$ and $t$ together have 6 unknowns (3 for rotation, 3 for translation). So with just 3 correspondences you have 6 equations for 6 unknowns — enough to solve the system. In practice you use more points for robustness.

This is exactly what `cv2.solvePnP()` does (covered in Notebook 09) — **P**erspective-**n**-**P**oint: given $n$ pairs of known 3D points and their observed 2D projections, find the pose.

## Recap

| Concept | Key equation / fact |
|---|---|
| 3 frames | World → Camera (extrinsics) → Image (intrinsics) |
| Pinhole projection | $u = f_x X_c/Z_c + c_x$, $v = f_y Y_c/Z_c + c_y$ |
| Intrinsic matrix K | $3 \times 3$; contains $f_x, f_y, c_x, c_y$; fixed per camera |
| Extrinsics [R\|t] | R = rotation ($3 \times 3$, orthogonal, det=1), t = translation |
| Full projection | $\mathbf{x} = K[R|t]\mathbf{X}$ |
| Homogeneous coords | Add 1; enables all transforms as matrix multiply; divide last element at end |
| Rodrigues vector | OpenCV's compact 3-element rotation representation; convert with `cv2.Rodrigues()` |
| 6D pose estimation | Given K and (world pt, image pt) pairs → solve for R and t |

---

**Next:** `07_camera_calibration.ipynb` — learn to measure K and distortion coefficients from chessboard images.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Projection from scratch
# ============================================================
# Given K = [[600, 0, 320], [0, 600, 240], [0, 0, 1]]
# Camera at origin, identity rotation.
#
# Project these 4 world points and report their pixel coordinates:
#   A: (0.0,  0.0, 1.0)  — straight ahead at 1m
#   B: (0.3,  0.0, 1.0)  — 0.3m right at 1m
#   C: (0.0,  0.3, 1.0)  — 0.3m down at 1m
#   D: (0.3,  0.3, 2.0)  — right+down at 2m
#
# Then explain: why does D appear closer to center than B projected at 1m?

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# K = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]], dtype=np.float64)
# R = np.eye(3)
# t = np.array([0.0, 0.0, 0.0])
#
# points = {'A': [0,0,1], 'B': [0.3,0,1], 'C': [0,0.3,1], 'D': [0.3,0.3,2]}
# for name, X in points.items():
#     Xc = R @ np.array(X) + t
#     u = K[0,0] * Xc[0] / Xc[2] + K[0,2]
#     v = K[1,1] * Xc[1] / Xc[2] + K[1,2]
#     print(f"{name}: world {X} → pixel ({u:.1f}, {v:.1f})")
#
# # D at 2m depth: u = 600*(0.3/2)+320 = 600*0.15+320 = 90+320 = 410
# # B at 1m depth: u = 600*(0.3/1)+320 = 600*0.3 +320 = 180+320 = 500
# # D appears closer to center (410 vs 500) because dividing by Z_c=2 halves the offset.
# # This IS the perspective effect — farther objects appear smaller/closer to center.

In [ ]:
# ============================================================
# EXERCISE 2: Validate rotation matrix properties
# ============================================================
# Write a function is_valid_rotation(R) that checks:
#   1. R.T @ R ≈ I (orthogonality)
#   2. det(R) ≈ +1 (proper rotation)
#   3. R.shape == (3, 3)
# Returns True if all pass, False otherwise.
#
# Test with:
#   - rotation_x(45)    — should be valid
#   - np.eye(3)         — should be valid (identity = no rotation)
#   - np.eye(3) * 2     — should be invalid (det = 8)
#   - np.zeros((3,3))   — should be invalid

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def is_valid_rotation(R, tol=1e-6):
#     if R.shape != (3, 3):
#         return False
#     ortho = np.allclose(R.T @ R, np.eye(3), atol=tol)
#     det_ok = abs(np.linalg.det(R) - 1.0) < tol
#     return ortho and det_ok
#
# tests = [
#     ('rotation_x(45)', rotation_x(45)),
#     ('np.eye(3)',       np.eye(3)),
#     ('eye * 2',        np.eye(3) * 2),
#     ('zeros',          np.zeros((3,3))),
#     ('rotation_z(90) @ rotation_y(30)', rotation_z(90) @ rotation_y(30)),
# ]
# for name, R in tests:
#     print(f"  {name:<40} valid={is_valid_rotation(R)}")

In [ ]:
# ============================================================
# EXERCISE 3: FOV and focal length relationship
# ============================================================
# A camera has image size 1280 x 720.
# Plot how the horizontal FOV changes as fx varies from 200 to 3000.
# Mark on the plot:
#   - fx that gives 90° FOV (wide angle)
#   - fx that gives 60° FOV (normal)
#   - fx that gives 30° FOV (telephoto)
#
# Formula: FOV_x = 2 * arctan(W / (2 * fx))  in degrees

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# W = 1280
# fx_vals = np.linspace(200, 3000, 500)
# fov_vals = 2 * np.degrees(np.arctan(W / (2 * fx_vals)))
#
# target_fovs = {90: 'Wide-angle', 60: 'Normal', 30: 'Telephoto'}
# plt.figure(figsize=(10, 5))
# plt.plot(fx_vals, fov_vals, 'b-', linewidth=2)
#
# for fov, label in target_fovs.items():
#     fx_target = W / (2 * np.tan(np.radians(fov / 2)))
#     plt.axhline(fov, color='gray', linestyle='--', alpha=0.5)
#     plt.axvline(fx_target, color='red', linestyle=':', alpha=0.7)
#     plt.annotate(f'{label}\nfx≈{fx_target:.0f}, FOV={fov}°',
#                  xy=(fx_target, fov), xytext=(fx_target+100, fov+3),
#                  fontsize=9, color='red')
#
# plt.xlabel('Focal length fx (pixels)')
# plt.ylabel('Horizontal FOV (degrees)')
# plt.title('FOV vs Focal Length (1280px wide image)')
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()